# Milestone 1: F1 Podium Prediction. Part 1: Data audit, cleaning and EDA

**Target:** `podium = 1` if the driver finishes in the top 3. **One row = one driver in one race.** We predict after qualifying and before the race starts, so only pre-race information can be used as features.

Sections: 1) setup, 2) loading and key audit, 3) EDA before cleaning, 4) cleaning and types, 5) duplicates, 6) Indy 500 and DNQs, 7) EDA after cleaning, 8) `load_clean()` and data dictionary.

## 1. Setup
Fixed random seed and one `DATA_DIR` variable that finds the CSVs on Kaggle or on a laptop, so the notebook runs with a single "Run All".

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")


import os, random
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SENT = "\\N"                    # the CSVs write a missing value as the text \N
KEYS = ["raceId", "driverId"]   # one row per driver per race
TABLES = ["circuits", "constructors", "constructor_results", "constructor_standings", "drivers",
          "driver_standings", "races", "results", "qualifying", "lap_times", "pit_stops",
          "sprint_results", "seasons", "status"]


def find_data_dir():
    """Works on Kaggle (/kaggle/input/...) and on a laptop (data/raw)."""
    folders = [Path("data/raw"), Path("../data/raw"), Path("data"), Path(".")]
    if Path("/kaggle/input").exists():
        folders = [p.parent for p in Path("/kaggle/input").rglob("races.csv")] + folders
    for folder in folders:
        if all((folder / f"{t}.csv").exists() for t in TABLES):
            return folder
    raise FileNotFoundError("F1 CSV files not found. Add the Kaggle dataset or put the CSVs in data/raw/")


def load_raw_tables(data_dir):
    """Read every CSV as plain text so nothing is guessed or silently changed."""
    raw = {}
    for t in TABLES:
        raw[t] = pd.read_csv(Path(data_dir) / f"{t}.csv", dtype=str, keep_default_na=False,
                             encoding_errors="replace")
    return raw


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("data folder:", DATA_DIR)

## 2. Loading the 14 tables and checking the keys

**Why this choice?** We read every file as text. The files write missing values as the text `\N`, and if pandas guessed the types, a numeric column with one `\N` would silently turn into text. Reading everything as text lets us count the `\N`s before changing anything, and the raw tables stay untouched (we only work on copies).

**Why check keys first?** Everything later is a join. If a primary key is duplicated, a join multiplies rows. If a foreign key points to a row that does not exist, a join silently drops rows. So we check that every primary key is unique and not empty, and that every foreign key exists in the table it points to.

In [ ]:
RAW = load_raw_tables(DATA_DIR)
display(pd.DataFrame({"rows": {t: len(df) for t, df in RAW.items()}, "columns": {t: df.shape[1] for t, df in RAW.items()}}))

In [ ]:
primary_keys = {
    "circuits": ["circuitId"], "constructors": ["constructorId"], "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"], "drivers": ["driverId"], "driver_standings": ["driverStandingsId"],
    "races": ["raceId"], "results": ["resultId"], "qualifying": ["qualifyId"],
    "lap_times": ["raceId", "driverId", "lap"], "pit_stops": ["raceId", "driverId", "stop"],
    "sprint_results": ["resultId"], "seasons": ["year"], "status": ["statusId"],
}
rows = []
for table, cols in primary_keys.items():
    df = RAW[table]
    empty = ((df[cols] == SENT) | (df[cols] == "")).any(axis=1).sum()
    duplicated = df.duplicated(cols, keep=False).sum()
    rows.append({"table": table, "primary key": "+".join(cols), "rows": len(df),
                 "empty keys": empty, "rows with duplicate key": duplicated, "OK": empty == 0 and duplicated == 0})
pk_audit = pd.DataFrame(rows)
display(pk_audit)

In [ ]:
foreign_keys = [  # (child table, child column, parent table, parent column)
    ("results", "raceId", "races", "raceId"), ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"), ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"), ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("races", "circuitId", "circuits", "circuitId"), ("races", "year", "seasons", "year"),
    ("sprint_results", "raceId", "races", "raceId"), ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"), ("sprint_results", "statusId", "status", "statusId"),
    ("driver_standings", "raceId", "races", "raceId"), ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"), ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"), ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"), ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"), ("pit_stops", "driverId", "drivers", "driverId"),
]
rows = []
for child, col, parent, pcol in foreign_keys:
    values = RAW[child][col]
    values = values[(values != SENT) & (values != "")]          # empty foreign keys are "missing", not "broken"
    orphans = (~values.isin(RAW[parent][pcol])).sum()
    rows.append({"child column": f"{child}.{col}", "points to": f"{parent}.{pcol}",
                 "orphans": orphans, "OK": orphans == 0})
fk_audit = pd.DataFrame(rows)
display(fk_audit)
print("primary keys OK:", int(pk_audit["OK"].sum()), "of", len(pk_audit), "| foreign keys OK:", int(fk_audit["OK"].sum()), "of", len(fk_audit))

**What this means.** A table marked OK can safely be the "one" side of a join. Any orphan row is reported, not repaired: we never invent a missing parent. Later we use inner joins on `raceId`, so a result with no race is dropped (it could not be placed in time anyway). Note that a surrogate key like `resultId` is unique by construction, so the real question for our table is whether `(raceId, driverId)` is unique. We check that in section 5.

## 3. EDA before cleaning

Everything here is computed from the raw tables, so we can compare with the cleaned data in section 7.

**3.1 Missing values (`\N`) per column.** We count the `\N` cells in every column and plot the worst columns and the overall distribution.

In [ ]:
rows = []
for table, df in RAW.items():
    for col in df.columns:
        n = (df[col] == SENT).sum()
        rows.append({"table": table, "column": col, "missing": n, "pct": 100 * n / len(df)})
missing = pd.DataFrame(rows)
with_missing = missing[missing["missing"] > 0].sort_values("pct", ascending=False)
print(len(with_missing), "of", len(missing), "columns contain at least one \\N")
display(with_missing.head(25))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top = with_missing.head(20)
ax[0].barh(top["table"] + "." + top["column"], top["pct"], color="tab:blue")
ax[0].invert_yaxis(); ax[0].set_xlabel("% of rows that are \\N"); ax[0].set_title("Top 20 columns by missing values")
ax[1].hist(missing["pct"], bins=20, color="tab:orange"); ax[1].set_yscale("log")
ax[1].set_xlabel("% missing in a column"); ax[1].set_ylabel("number of columns (log scale)")
ax[1].set_title("Missingness over all columns")
plt.show()

**What we see.** Most columns have no `\N` at all and a few have a lot. The big ones are the qualifying times `q2` and `q3` (a driver knocked out in Q1 never drives Q2 or Q3, and before 2006 there was no Q2 or Q3), the fastest-lap columns in old seasons (not recorded), and the newer schedule columns in `races`. The reason differs per column, so in cleaning we only turn `\N` into `NaN` and we do **not** fill anything in. Filling in is done later by the modeling team on the training years only.

**3.2 Field size and podium rate per season.** The podium always has 3 places, so the share of drivers on the podium is about 3 divided by the number of cars (the field size).

In [ ]:
def num(s):
    return pd.to_numeric(s.replace(SENT, np.nan), errors="coerce")

res_raw = RAW["results"][["raceId", "driverId", "positionOrder"]].copy()
res_raw["positionOrder"] = num(res_raw["positionOrder"])
races_raw = RAW["races"][["raceId", "year"]].copy()
races_raw["year"] = num(races_raw["year"])
res_raw = res_raw.merge(races_raw, on="raceId")
res_raw["podium"] = res_raw["positionOrder"] <= 3

season_raw = res_raw.groupby("year").agg(rows=("driverId", "size"), races=("raceId", "nunique"), podium_rate=("podium", "mean"))
season_raw["rows_per_race"] = season_raw["rows"] / season_raw["races"]
season_raw["unique_drivers_per_race"] = res_raw.groupby(["year", "raceId"])["driverId"].nunique().groupby("year").mean()
season_raw["3_divided_by_field"] = 3 / season_raw["rows_per_race"]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(season_raw.index, season_raw["rows_per_race"], label="result rows per race")
ax[0].plot(season_raw.index, season_raw["unique_drivers_per_race"], "--", label="different drivers per race")
ax[0].set_title("Entrants per race (raw)"); ax[0].set_xlabel("season"); ax[0].legend()
ax[1].plot(season_raw.index, season_raw["podium_rate"], label="observed podium rate")
ax[1].plot(season_raw.index, season_raw["3_divided_by_field"], "--", label="3 / field size")
ax[1].set_title("Podium rate per season (raw)"); ax[1].set_xlabel("season"); ax[1].legend()
plt.show()
print("overall podium rate (raw rows):", round(res_raw["podium"].mean(), 3))

**What we see.** The field size changes a lot over the years (big and uneven fields in the 1950s and in the 1980s-90s, about 20 cars today), so the podium rate changes too: about 15% with 20 cars, much lower with 30+. In the early years there are more result rows per race than different drivers. That gap is the first sign of duplicate rows (section 5). So accuracy is not a good metric for this problem, and the class balance will be different in our train, validation and test years.

**3.3 Qualifying coverage per season.**

In [ ]:
quali_raw = RAW["qualifying"]
flags = quali_raw.assign(has_q1=(quali_raw["q1"] != SENT) & (quali_raw["q1"] != ""),
                         has_q3=(quali_raw["q3"] != SENT) & (quali_raw["q3"] != ""))
flags = flags.groupby(KEYS)[["has_q1", "has_q3"]].max().reset_index()
flags["has_quali_row"] = True

cov = res_raw[KEYS + ["year"]].merge(flags, on=KEYS, how="left")
cov[["has_quali_row", "has_q1", "has_q3"]] = cov[["has_quali_row", "has_q1", "has_q3"]].fillna(False).astype(bool)
coverage = cov.groupby("year")[["has_quali_row", "has_q1", "has_q3"]].mean()
first_year = coverage.index[coverage["has_quali_row"] > 0].min()

ax = coverage.plot(figsize=(11, 4.5), lw=2)
ax.axvline(first_year, color="k", ls=":")
ax.set_title(f"Share of result rows with qualifying data (first season: {int(first_year)})")
ax.set_xlabel("season"); ax.set_ylabel("share of rows")
plt.show()
print("first season with qualifying rows:", int(first_year))

**What we see.** There is no qualifying data at all before the first season printed above, then coverage grows. Q1 appears first and Q2/Q3 only in the knockout format (2006 onwards).

**Why we do not fill in old qualifying times:** for the old seasons the data is not "randomly missing", the session is simply not in the dataset (and the formats were different, so a 1962 time is not comparable with a 2015 time). Filling it with an average would invent a feature for half of F1 history. Instead `load_clean()` keeps a `has_qualifying` flag, and the grid position (always available) is the fallback for old seasons. The modeling team can also decide to train only on the seasons with qualifying data.

## 4. Cleaning and types

**Why this choice?**
* `\N` is text. In a number column it turns the whole column into text, so we cannot average it, and comparing text like `"10" < "2"` is wrong. We replace `\N` (and blank cells, which turned up in `q2`/`q3`) by a real missing value, and we convert columns to proper types: ids and counts to integers that can hold missing values (`Int64`), dates to datetime, and lap times like `1:27.452` to seconds (`87.452`) so they can be compared and subtracted.
* We only convert, we do not fill in. Why a value is missing differs by column (see 3.1), so one rule would be wrong, and filling with an average from all years could leak information from the future.

In [ ]:
INT_COLS = {
    "races": ["year", "round"], "seasons": ["year"], "drivers": ["number"],
    "constructor_standings": ["position", "wins"], "driver_standings": ["position", "wins"],
    "results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap", "rank"],
    "qualifying": ["number", "position"],
    "lap_times": ["lap", "position", "milliseconds"],
    "pit_stops": ["stop", "lap", "milliseconds"],
    "sprint_results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap"],
}
FLOAT_COLS = {
    "circuits": ["lat", "lng", "alt"],
    "constructor_results": ["points"], "constructor_standings": ["points"], "driver_standings": ["points"],
    "results": ["points", "fastestLapSpeed"], "sprint_results": ["points", "fastestLapSpeed"],
}
DATE_COLS = {
    "races": ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"],
    "drivers": ["dob"],
}
TIME_COLS = {   # text lap times -> seconds (old column: new column)
    "results": {"fastestLapTime": "fastestLapTime_s"},
    "sprint_results": {"fastestLapTime": "fastestLapTime_s"},
    "qualifying": {"q1": "q1_s", "q2": "q2_s", "q3": "q3_s"},
    "lap_times": {"time": "time_s"},
    "pit_stops": {"duration": "duration_s"},
}


def to_seconds(s):
    """'1:27.452' -> 87.452 (also handles '59.3' and '1:02:03.5'). Anything else -> NaN."""
    parts = s.astype("string").str.extract(r"^(?:(?:(\d+):)?(\d+):)?(\d+(?:\.\d+)?)$")
    hours = pd.to_numeric(parts[0]).fillna(0)
    minutes = pd.to_numeric(parts[1]).fillna(0)
    seconds = pd.to_numeric(parts[2])
    return (hours * 3600 + minutes * 60 + seconds).astype("float64")


def clean_tables(raw):
    """Return cleaned copies of all tables. The raw tables are not touched."""
    clean = {}
    for name, df in raw.items():
        d = df.replace(SENT, np.nan)                      # \N -> NaN
        d = d.replace(r"^\s*$", np.nan, regex=True)       # blank cells are missing too
        for c in d.columns:                               # ids and counts -> integers (can hold NaN)
            if c.endswith("Id") or c in INT_COLS.get(name, []):
                d[c] = pd.to_numeric(d[c], errors="coerce").round().astype("Int64")
        for c in FLOAT_COLS.get(name, []):
            if c in d.columns:
                d[c] = pd.to_numeric(d[c], errors="coerce")
        for c in DATE_COLS.get(name, []):                 # text -> real dates
            if c in d.columns:
                d[c] = pd.to_datetime(d[c], format="%Y-%m-%d", errors="coerce")
        for old, new in TIME_COLS.get(name, {}).items():  # text lap times -> seconds
            if old in d.columns:
                d[new] = to_seconds(d[old])
                d = d.drop(columns=[old])
        clean[name] = d
    return clean

In [ ]:
CLEAN = clean_tables(RAW)

# the lap-time converter on a few examples
test = pd.Series(["1:27.452", "59.3", "1:02:03.5", np.nan, "oops"])
print(dict(zip(test.fillna("NaN"), to_seconds(test).round(3))))

# no \N left anywhere
left = 0
for name, df in CLEAN.items():
    for col in df.columns:
        if df[col].dtype == object or str(df[col].dtype) in ("string", "str"):
            left += (df[col] == SENT).sum()
print("\\N left in the cleaned tables:", left)

pairs = [("results", "grid", "grid"), ("results", "positionOrder", "positionOrder"), ("results", "points", "points"),
         ("results", "fastestLapTime", "fastestLapTime_s"), ("qualifying", "q3", "q3_s"),
         ("races", "date", "date"), ("drivers", "dob", "dob")]
display(pd.DataFrame([(t, f"{a} -> {b}", str(RAW[t][a].dtype), str(CLEAN[t][b].dtype)) for t, a, b in pairs],
                     columns=["table", "column", "type before", "type after"]))

# results joined with the race info and the status text (needed for the next sections)
RES = CLEAN["results"].merge(CLEAN["races"][["raceId", "year", "round", "date", "circuitId"]], on="raceId", how="inner")
RES = RES.merge(CLEAN["status"], on="statusId", how="left")
print("result rows:", len(RES), "(raw:", len(RAW["results"]), ")")

### What does `grid = 0` mean?
The project notes say `0` is probably a pit-lane start. We check two ideas:
* **Pit-lane start:** the driver did race (laps > 0) but had no grid slot.
* **No grid slot because he did not start:** DNQ or withdrawn drivers, mostly in old seasons.

In [ ]:
g = RES.copy()
g["grid_zero"] = (g["grid"] == 0).fillna(False).astype(bool)
g["decade"] = (g["year"] // 10 * 10).astype(int)
g["ran"] = g["laps"].fillna(0) > 0

table = g.groupby("decade").agg(rows=("grid", "size"), grid_zero=("grid_zero", "sum"))
table["grid_zero_and_raced"] = g[g.grid_zero & g.ran].groupby("decade").size()
table["grid_zero_no_laps"] = g[g.grid_zero & ~g.ran].groupby("decade").size()
table = table.fillna(0).astype(int)
display(table)

table[["grid_zero_and_raced", "grid_zero_no_laps"]].plot(kind="bar", stacked=True, figsize=(9, 4), color=["tab:green", "tab:red"])
plt.title("Rows with grid = 0, by decade"); plt.ylabel("rows"); plt.show()

old, new = g[g.grid_zero & (g.year < 1990)], g[g.grid_zero & (g.year >= 2000)]
print("before 1990:", len(old), "rows | share that completed laps:", round(old["ran"].mean(), 2), "| positionText:", old["positionText"].value_counts().head(4).to_dict())
print("2000+      :", len(new), "rows | share that completed laps:", round(new["ran"].mean(), 2), "| top statuses:", new["status"].value_counts().head(4).to_dict())

**Conclusion.** `grid = 0` means "no grid slot", and there are two kinds of rows behind it. In modern seasons these are drivers who raced from the **pit lane** (after a penalty, a car change, or missing the grid). In old seasons they are mostly drivers who **did not start** (failed to qualify or withdrew) or whose slot was not recorded.

**Why this choice for the table:** `0` must never be read as "better than pole position". We remove the non-starters first (section 6), and what is left with `grid = 0` is flagged as `pitlane_start`. We keep the original `grid` column unchanged.

## 5. One row per driver per race (duplicates)

**Why we need exactly one row per `(raceId, driverId)`:** every row is one question for the model: "will this driver finish on the podium in this race?" If a driver has two rows, they can have different labels and different features (two cars, two grid slots), so the model would be asked the same question twice with different answers. It also counts that driver twice in every rate. Two rows for the same driver also means the same race can end up in two different splits.

**Why duplicates exist:** until 1957 a driver could take over a teammate's car during the race (a *shared drive*) and the points were split. The data has one result row per car, so the same driver gets two rows in one race.

In [ ]:
dup_all = RES.duplicated(KEYS, keep=False)        # every row that belongs to a duplicate group
extra = RES.duplicated(KEYS, keep="first")        # the rows beyond the first one
print("duplicate groups (driver-races with more than one row):", RES[dup_all].groupby(KEYS).ngroups)
print("rows in those groups:", int(dup_all.sum()), "| extra rows to remove:", int(extra.sum()))

per_year = extra.groupby(RES["year"]).sum().reindex(range(1950, 2025), fill_value=0)
print("extra rows in 1950-1957:", int(per_year.loc[1950:1957].sum()), "of", int(per_year.sum()))
plt.figure(figsize=(12, 3.8))
plt.bar(per_year.index, per_year.values, color="tab:red")
plt.title("Duplicate (raceId, driverId) rows per season"); plt.xlabel("season"); plt.ylabel("extra rows"); plt.show()

examples = RES[dup_all].merge(CLEAN["drivers"][["driverId", "surname"]], on="driverId")
examples = examples.merge(CLEAN["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId", how="left")
display(examples.sort_values(["year", "raceId", "surname"])[["year", "raceId", "surname", "constructor", "number", "grid", "positionOrder", "status"]].head(10))

### The rule for choosing which row to keep
There are two separate questions, and they follow different rules:

1. **Which row do we keep?** Its `grid`, `constructorId` and `number` become features, so we may only use facts that are known **before the race**: first a car that really started (`grid > 0`), then the lowest grid slot, then `constructorId` and car number as arbitrary but fixed tie-breaks. `resultId` is only the very last tie-break, for rows that look identical before the race. We do **not** use `positionOrder` here, because choosing the row by its finishing position would let the race result decide the features.
2. **What is the label?** The label is the outcome, so it may use the result. `podium = 1` if the driver's **best** finish over all his rows in that race is 1, 2 or 3 (a driver who took over a podium car did get that finish). For a normal driver with one row this is just `positionOrder <= 3`.

**Limitation:** when two cars both started, the pre-race columns cannot tell which one was the driver's own, so the kept `grid` may be the teammate's slot. This only affects shared drives, which are a tiny part of the data (checked below) and only exist before 1958, so the validation and test years are not touched. These rows are flagged with `shared_drive`.

In [ ]:
def flag_indy500(races, circuits):
    """Mark the Indianapolis 500 rounds that counted for the championship (1950-1960).
    The same circuit hosted a real F1 US Grand Prix in 2000-2007, so we also check the year."""
    races = races.copy()
    indy_ids = circuits.loc[circuits["name"].str.contains("Indianapolis", na=False), "circuitId"]
    by_name = races["name"].str.contains("Indianapolis 500", na=False)
    by_place = races["circuitId"].isin(indy_ids) & races["year"].between(1950, 1960)
    races["is_indy500"] = (by_name | by_place).astype(bool)
    return races


def find_non_starters(res):
    """Drivers who did not qualify / withdrew. Only uses facts known before the race."""
    by_code = res["positionText"].isin(["F", "W"])      # F = failed to qualify, W = withdrew
    by_status = res["status"].str.contains("did not (?:pre)?qualify|not qualified|withdrew|107%",
                                           case=False, regex=True, na=False)
    return (by_code | by_status).astype(bool)


def keep_one_row_per_driver(res):
    """One row per (raceId, driverId).
    WHICH row we keep uses only facts known before the race (never the finishing position):
      1) a car that really started (grid > 0), 2) the lowest grid slot, 3) constructorId, car number,
      4) resultId only as a last resort (rows that look identical before the race).
    The LABEL is different: podium = the driver's best finish over all his rows <= 3."""
    res = res.reset_index(drop=True).copy()
    res["n_raw_rows"] = res.groupby(KEYS)["resultId"].transform("size")
    res["best_positionOrder"] = res.groupby(KEYS)["positionOrder"].transform("min")
    res["started"] = res["grid"].fillna(0) > 0
    res["grid_for_sort"] = res["grid"].where(res["started"])
    res = res.sort_values(KEYS + ["started", "grid_for_sort", "constructorId", "number", "resultId"],
                          ascending=[True, True, False, True, True, True, True], kind="mergesort")
    kept = res.drop_duplicates(KEYS, keep="first")
    dropped = res.drop(kept.index).drop(columns=["started", "grid_for_sort"])
    kept = kept.drop(columns=["started", "grid_for_sort"]).copy()
    kept["shared_drive"] = kept["n_raw_rows"] > 1
    kept["podium"] = (kept["best_positionOrder"] <= 3).astype("int8")
    return kept.reset_index(drop=True), dropped

In [ ]:
RES_ONE, DROPPED = keep_one_row_per_driver(RES)
assert not RES_ONE.duplicated(KEYS).any()
assert len(RES_ONE) + len(DROPPED) == len(RES)
print(f"rows: {len(RES)} -> {len(RES_ONE)} (removed {len(DROPPED)}); every (raceId, driverId) pair is still there")

shared = RES_ONE[RES_ONE["shared_drive"]]
own_row_label = (shared["positionOrder"] <= 3).astype(int)
print(f"shared-drive rows: {len(shared)} ({len(shared) / len(RES_ONE):.2%} of all rows), seasons {shared['year'].min()}-{shared['year'].max()}")
print("podium label if we only used the kept row:", int(own_row_label.sum()),
      "| with the driver's best row (used):", int(shared["podium"].sum()),
      "| rows where the two differ:", int((own_row_label != shared["podium"]).sum()))

## 6. Indianapolis 500 and DNQ / non-starters

### Indianapolis 500 (1950-1960)
This race counted for the championship, but almost no F1 drivers entered it, so we **exclude it**.

**Why this choice?** It is a different kind of race (oval, 33 cars, different grid), its drivers are mostly Americans who never raced elsewhere in F1, so it would distort the home-race question (Q2) and the driver form features. We find it by race name and year (not by circuit), because the same Indianapolis circuit hosted a normal F1 US Grand Prix in 2000-2007 that we keep.

In [ ]:
RACES = flag_indy500(CLEAN["races"], CLEAN["circuits"])
print("Indy 500 rounds:", int(RACES["is_indy500"].sum()), "| seasons:", sorted(RACES.loc[RACES.is_indy500, "year"].unique().tolist()))
assert RACES.loc[RACES.year.between(2000, 2007), "is_indy500"].sum() == 0     # the modern US GP stays

RES_ONE = RES_ONE.merge(RACES[["raceId", "is_indy500"]], on="raceId", how="left")
d = RES_ONE.merge(CLEAN["drivers"][["driverId", "nationality"]], on="driverId", how="left")
d["american"] = d["nationality"] == "American"
us_circuits = CLEAN["circuits"].loc[CLEAN["circuits"]["country"].isin(["USA", "United States"]), "circuitId"]
home_rows = d[d["american"] & d["circuitId"].isin(us_circuits)]
print("rows of American drivers at US circuits:", len(home_rows), "| of which Indy 500 rows:", int(home_rows["is_indy500"].sum()),
      f"({home_rows['is_indy500'].mean():.0%})")
print("drivers who only ever appear at the Indy 500:", int(d.groupby("driverId")["is_indy500"].all().sum()))

by_year = d[d.year.between(1950, 1960) & d.american].groupby(["year", "is_indy500"]).size().unstack(fill_value=0)
by_year.plot(kind="bar", stacked=True, figsize=(9, 3.8))
plt.title("American driver entries per season, 1950-1960"); plt.legend(["other F1 races", "Indy 500"]); plt.ylabel("rows"); plt.show()

### DNQ / non-starters
In the years with more cars than grid places, slow drivers *failed to (pre)qualify* but still have a row in `results`. Withdrawn drivers are the same. We **separate them from the modeling table**.

**Why this choice?** The model predicts after the grid is set, so a driver who is not on the grid is not part of the question, and their label is always 0. They also make the podium rate artificially low in the big-field years. They never have a podium finish, so removing them does not remove any positives. We find them only with facts known before the race (failed to qualify, withdrew, 107% rule). We do not use the number of laps or the race status of drivers who raced, because dropping rows based on how the race went would use the outcome.

In [ ]:
RES_ONE["is_non_starter"] = find_non_starters(RES_ONE)
ns = RES_ONE[RES_ONE["is_non_starter"] & ~RES_ONE["is_indy500"]]
print("non-starter rows (outside Indy 500):", len(ns), "of", int((~RES_ONE["is_indy500"]).sum()))
print("by decade:", {int(k): int(v) for k, v in ns.groupby(ns["year"] // 10 * 10).size().items()})
print("positionText:", ns["positionText"].value_counts().head(4).to_dict(), "| status:", ns["status"].value_counts().head(3).to_dict())
print("non-starters with a podium finish:", int((ns["podium"] == 1).sum()), "(should be 0)")

RES_FINAL = RES_ONE[~RES_ONE["is_indy500"] & ~RES_ONE["is_non_starter"]].reset_index(drop=True)
print("modeling population:", len(RES_FINAL), "rows")

dec = RES_ONE[~RES_ONE["is_indy500"]].copy()
dec["decade"] = dec["year"] // 10 * 10
rate = pd.DataFrame({"with non-starters": dec.groupby("decade")["podium"].mean(),
                     "starters only": dec[~dec["is_non_starter"]].groupby("decade")["podium"].mean()})
rate.plot(kind="bar", figsize=(9, 3.8)); plt.title("Podium rate per decade"); plt.ylabel("podium rate"); plt.show()

## 7. EDA after cleaning and before/after comparison

We repeat the metrics from section 3 on the cleaned data (`RES_FINAL`) and put them next to the raw ones.

**First check: did turning `\N` into `NaN` lose or invent anything?** For every column that had missing values we compare the number of missing cells before (`\N` or blank) with the number of `NaN` after. They must be equal.

In [ ]:
rows = []
for table, df in RAW.items():
    for col in df.columns:
        before = ((df[col] == SENT) | (df[col].str.strip() == "")).sum()
        if before == 0:
            continue
        new_col = TIME_COLS.get(table, {}).get(col, col)
        after = CLEAN[table][new_col].isna().sum()
        rows.append({"table": table, "column": col, "missing before": before, "NaN after": after, "difference": after - before})
check = pd.DataFrame(rows)
print("columns checked:", len(check), "| columns with a difference:", int((check["difference"] != 0).sum()))
display(check[check["difference"] != 0])

top = check.sort_values("missing before", ascending=False).head(12)
x = np.arange(len(top))
plt.figure(figsize=(11, 4))
plt.bar(x - 0.2, top["missing before"], 0.4, label="missing before (\\N or blank)")
plt.bar(x + 0.2, top["NaN after"], 0.4, label="NaN after")
plt.xticks(x, top["table"] + "." + top["column"], rotation=70, ha="right"); plt.yscale("log"); plt.legend()
plt.title("Missing values before and after cleaning"); plt.show()

In [ ]:
# field size and podium rate per season: raw vs cleaned
season_clean = RES_FINAL.groupby("year").agg(rows=("driverId", "size"), races=("raceId", "nunique"), podium_rate=("podium", "mean"))
season_clean["rows_per_race"] = season_clean["rows"] / season_clean["races"]

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
ax[0, 0].plot(season_raw.index, season_raw["rows_per_race"], label="raw rows per race")
ax[0, 0].plot(season_clean.index, season_clean["rows_per_race"], label="clean rows per race (starters)")
ax[0, 0].set_title("Field size per season"); ax[0, 0].legend()
ax[0, 1].plot(season_raw.index, season_raw["podium_rate"], label="raw")
ax[0, 1].plot(season_clean.index, season_clean["podium_rate"], label="clean")
ax[0, 1].plot(season_clean.index, 3 / season_clean["rows_per_race"], "k:", label="3 / clean field size")
ax[0, 1].set_title("Podium rate per season"); ax[0, 1].legend()

podium_rows_raw = res_raw.groupby("raceId")["podium"].sum().value_counts().sort_index()
podium_rows_clean = RES_FINAL.groupby("raceId")["podium"].sum().value_counts().sort_index()
pd.concat([podium_rows_raw.rename("raw"), podium_rows_clean.rename("clean")], axis=1).fillna(0).plot(kind="bar", ax=ax[1, 0])
ax[1, 0].set_title("Podium rows per race (3 expected)"); ax[1, 0].set_xlabel("podium rows in a race"); ax[1, 0].set_ylabel("races")

lap = CLEAN["results"]["fastestLapTime_s"].dropna()
ax[1, 1].hist(lap.clip(30, 200), bins=60, color="tab:green")
ax[1, 1].set_title("fastestLapTime_s after conversion (seconds)")
plt.show()

print("races without exactly 3 podium rows after cleaning:", int((podium_rows_clean.drop(3, errors="ignore")).sum()), "of", int(podium_rows_clean.sum()))
for col, df in [("fastestLapTime_s", CLEAN["results"]), ("q1_s", CLEAN["qualifying"]), ("q3_s", CLEAN["qualifying"])]:
    s = df[col].dropna()
    print(f"{col:17s} n={len(s):6d}  min={s.min():7.2f}  median={s.median():7.2f}  max={s.max():8.2f}")

In [ ]:
# what each cleaning rule removed
def podium_rows(df):
    return int((df["positionOrder"] <= 3).sum())

removed_indy = RES_ONE[RES_ONE["is_indy500"]]
removed_ns = RES_ONE[~RES_ONE["is_indy500"] & RES_ONE["is_non_starter"]]
summary = pd.DataFrame([
    ("typed results (start)", len(RES), podium_rows(RES)),
    ("removed: duplicate rows", len(DROPPED), podium_rows(DROPPED)),
    ("removed: Indy 500", len(removed_indy), podium_rows(removed_indy)),
    ("removed: non-starters", len(removed_ns), podium_rows(removed_ns)),
    ("modeling population", len(RES_FINAL), podium_rows(RES_FINAL)),
], columns=["step", "rows", "rows with positionOrder <= 3"])
display(summary)
print("podium label = 1 in the final table:", int(RES_FINAL["podium"].sum()))

**What the cleaning achieved**
* **Missing values:** the number of missing cells before and after is the same in every column (the table above is empty when that is true), so nothing was lost or invented. The lap times are now numbers in a sensible range (tens of seconds, no conversion errors).
* **Duplicates:** the rows per race now match the number of different drivers, and almost every race has exactly 3 podium rows. The few races that do not are shared-drive cases where two drivers were credited with the same podium car.
* **Podium rate:** the change is concentrated in the early decades (duplicates, the Indy 500 and non-starters). Recent seasons, which are our validation and test years, are practically unchanged. The non-starter rule removes no podium rows, and the duplicates are only removed as extra rows, never as a driver-race.

## 8. `load_clean()` and the data dictionary

**Why one function?** If every teammate cleaned the data in their own way (one keeps the Indy 500, another drops DNQs), the row counts and scores would no longer be comparable. `load_clean()` runs all the steps above in the same order, so everyone starts from identical tables.

It returns:
* `model_base`: one row per driver per race with **pre-race columns only** plus the target `podium`.
* `outcomes`: the **post-race** columns of the same rows (status, points, positions, laps...). They are used for the data-engineering questions and must **not** be used as features.
* `tables`: all other cleaned tables. The standings tables are post-race snapshots, so use the previous round before using them as features.
* `meta`: how many rows each step removed.

The function stops with an error if a driver-race has more than one row or if a post-race column ends up in `model_base`. It does not fill in or scale anything, so nothing can be fitted on the test years.

In [ ]:
PRE_RACE_COLS = ["raceId", "driverId", "constructorId", "year", "round", "date", "circuitId", "number",
                 "grid", "pitlane_start", "has_qualifying", "qual_position", "q1_s", "q2_s", "q3_s"]
OUTCOME_COLS = ["raceId", "driverId", "resultId", "positionOrder", "podium", "positionText", "position",
                "points", "laps", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime_s",
                "fastestLapSpeed", "statusId", "status", "n_raw_rows", "shared_drive", "best_positionOrder"]
POST_RACE_COLS = {"positionOrder", "positionText", "position", "points", "laps", "time", "milliseconds",
                  "fastestLap", "rank", "fastestLapTime_s", "fastestLapSpeed", "statusId", "status",
                  "resultId", "n_raw_rows", "shared_drive", "best_positionOrder"}


def load_clean(data_dir=None, exclude_indy500=True, drop_non_starters=True, exclude_shared_drives=False):
    """Raw CSVs -> clean tables. Everybody on the team should start from this function.
    model_base = pre-race columns + target `podium` (one row per driver per race)
    outcomes   = post-race columns (status, points, ...) - never use these as features."""
    random.seed(SEED)
    np.random.seed(SEED)
    tables = clean_tables(load_raw_tables(data_dir or find_data_dir()))
    tables["races"] = flag_indy500(tables["races"], tables["circuits"])

    res = tables["results"].merge(tables["races"][["raceId", "year", "round", "date", "circuitId", "is_indy500"]],
                                  on="raceId", how="inner")
    res = res.merge(tables["status"], on="statusId", how="left")
    meta = {"raw_result_rows": len(tables["results"]), "after_race_join": len(res)}

    if exclude_indy500:
        meta["indy500_rows_removed"] = int(res["is_indy500"].sum())
        res = res[~res["is_indy500"]]
    res, dropped = keep_one_row_per_driver(res)
    meta["duplicate_rows_removed"] = len(dropped)
    res["is_non_starter"] = find_non_starters(res)
    if drop_non_starters:
        meta["non_starter_rows_removed"] = int(res["is_non_starter"].sum())
        res = res[~res["is_non_starter"]]
    if exclude_shared_drives:
        meta["shared_drive_rows_removed"] = int(res["shared_drive"].sum())
        res = res[~res["shared_drive"]]
    res = res.reset_index(drop=True)
    meta["final_rows"] = len(res)

    quali = tables["qualifying"].sort_values("qualifyId")
    meta["qualifying_duplicates_dropped"] = int(quali.duplicated(KEYS, keep="last").sum())
    quali = quali.drop_duplicates(KEYS, keep="last").rename(columns={"position": "qual_position"})
    quali = quali[KEYS + ["qual_position", "q1_s", "q2_s", "q3_s"]]

    res["pitlane_start"] = (res["grid"] == 0).fillna(False).astype(bool)
    base = res.merge(quali, on=KEYS, how="left", indicator=True)
    base["has_qualifying"] = base["_merge"] == "both"

    model_base = base[PRE_RACE_COLS].copy()
    model_base["podium"] = base["podium"].values
    outcomes = base[OUTCOME_COLS].copy()

    # safety checks
    assert not model_base.duplicated(KEYS).any(), "more than one row for a driver in a race"
    assert model_base[KEYS].notna().all().all(), "missing key"
    assert not (POST_RACE_COLS & set(model_base.columns)), "post-race column in the base table"
    return {"model_base": model_base, "outcomes": outcomes, "tables": tables, "meta": meta}

In [ ]:
CL = load_clean(DATA_DIR)
model_base, outcomes = CL["model_base"], CL["outcomes"]
print(CL["meta"])
display(model_base.head())

# same result as the step-by-step version above
assert len(model_base) == len(RES_FINAL) and model_base["podium"].sum() == RES_FINAL["podium"].sum()
print(f"{len(model_base)} rows, {model_base['raceId'].nunique()} races, seasons {model_base['year'].min()}-{model_base['year'].max()}, podium rate {model_base['podium'].mean():.1%}")

split = np.select([model_base["year"] <= 2019, model_base["year"].between(2020, 2021)], ["train (<=2019)", "validation (2020-21)"], "test (>=2022)")
display(model_base.groupby(split)["podium"].agg(rows="size", podium_rate="mean").round(3))

In [ ]:
descriptions = {
    "raceId": ("key", "race id (links to races)"), "driverId": ("key", "driver id (links to drivers)"),
    "constructorId": ("pre-race", "team of the kept row"), "year": ("pre-race", "season"), "round": ("pre-race", "round number in the season"),
    "date": ("pre-race", "race date"), "circuitId": ("pre-race", "circuit (links to circuits)"), "number": ("pre-race", "car number"),
    "grid": ("pre-race", "starting slot after penalties; 0 = no grid slot (pit-lane start for drivers who started)"),
    "pitlane_start": ("pre-race", "True if grid = 0 (non-starters were already removed)"),
    "has_qualifying": ("pre-race", "True if the driver has a qualifying row"),
    "qual_position": ("pre-race", "qualifying position (NaN when there is no qualifying data)"),
    "q1_s": ("pre-race", "Q1 best lap in seconds"), "q2_s": ("pre-race", "Q2 best lap in seconds (NaN if knocked out in Q1 or before 2006)"),
    "q3_s": ("pre-race", "Q3 best lap in seconds (NaN if knocked out earlier or before 2006)"),
    "podium": ("TARGET", "1 if the driver's best finish in the race is top 3"),
    "resultId": ("outcome", "id of the kept result row"), "positionOrder": ("outcome", "final ranking of every entrant"),
    "positionText": ("outcome", "position or letter code (R, D, E, W, F, N)"), "position": ("outcome", "classified position"),
    "points": ("outcome", "championship points"), "laps": ("outcome", "laps completed"), "time": ("outcome", "race time text"),
    "milliseconds": ("outcome", "race time in ms"), "fastestLap": ("outcome", "lap of the fastest lap"), "rank": ("outcome", "fastest-lap rank"),
    "fastestLapTime_s": ("outcome", "fastest lap in seconds"), "fastestLapSpeed": ("outcome", "fastest lap speed"),
    "statusId": ("outcome", "why the race ended (links to status)"), "status": ("outcome", "status text"),
    "n_raw_rows": ("audit", "number of raw result rows of this driver in this race"),
    "shared_drive": ("audit", "True if the driver had more than one raw row (car swap, before 1958)"),
    "best_positionOrder": ("audit", "best positionOrder over the driver's rows (used for the label)"),
}
dictionary = pd.DataFrame([(tbl, col, str(df[col].dtype), *descriptions[col], round(100 * df[col].isna().mean(), 1))
                           for tbl, df in [("model_base", model_base), ("outcomes", outcomes)] for col in df.columns],
                          columns=["table", "column", "type", "role", "meaning", "% missing"])
display(dictionary)

dictionary.to_csv(OUTPUT_DIR / "data_dictionary.csv", index=False)
model_base.to_csv(OUTPUT_DIR / "model_base.csv", index=False)
outcomes.to_csv(OUTPUT_DIR / "outcomes.csv", index=False)

### Notes for the team
* Features come from `model_base` (and from earlier races). Columns marked `outcome` or `audit` are never features.
* Standings tables are after-the-race: shift them to the previous round. Rolling form uses `shift(1)`.
* Fit imputers, scalers and encoders on the training years only (<= 2019).
* Shared-drive rows (about 0.3% of rows, all before 1958) are kept and flagged. `load_clean(exclude_shared_drives=True)` removes them if the modeling team prefers.